# Adapted from https://milvus.io/docs/full-text-search.md

In [1]:
%%capture
from pymilvus import MilvusClient, DataType, Function, FunctionType

client = MilvusClient("./milvus_demo.db")

In [2]:
schema = client.create_schema()

schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=2048, enable_analyzer=True)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048, 'enable_analyzer': True}}, {'name': 'sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [3]:
bm25_function = Function(
    name="text_bm25_emb", # Function name
    input_field_names=["text"], # Name of the VARCHAR field containing raw text data
    output_field_names=["sparse"], # Name of the SPARSE_FLOAT_VECTOR field reserved to store generated embeddings
    function_type=FunctionType.BM25, # Set to `BM25`
)

schema.add_function(bm25_function)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048, 'enable_analyzer': True}}, {'name': 'sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>, 'is_function_output': True}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'text_bm25_emb', 'description': '', 'type': <FunctionType.BM25: 1>, 'input_field_names': ['text'], 'output_field_names': ['sparse'], 'params': {}}]}

In [4]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="sparse",

    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.2,
        "bm25_b": 0.75
    }

)

In [5]:
# Drop existing collection with the same name if it exists
if client.has_collection("nyu_rt_docs"):
    client.drop_collection("nyu_rt_docs")

In [6]:
client.create_collection(
    collection_name='nyu_rt_docs', 
    schema=schema, 
    index_params=index_params
)

## Set up chunker for use

In [7]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

converter = DocumentConverter()
chunker = HybridChunker()

In [8]:
import pickle

with open("docs.pickle", "rb") as f:
    docs = pickle.load(f)

In [ ]:
"https://services.rt.nyu.edu/docs/hpc/getting_started/intro/" in docs

In [ ]:
from tqdm import tqdm

for guide in tqdm(docs):
    try:
        doc = converter.convert(source=guide).document
        chunk_iter = chunker.chunk(dl_doc=doc)
        texts = [chunk.text for chunk in chunker.chunk(doc)]

        for text in texts:
            client.insert('nyu_rt_docs', [{'text': text},])
    except:
        pass

In [12]:
query = "How do I request an HPC account?"

search_params = {
    'params': {'drop_ratio_search': 0.2},
} 

num_retreive = 3

retreived_chunks = client.search(
    collection_name='nyu_rt_docs', 
    data=[query],
    anns_field='sparse',
    output_fields=['text'], # Fields to return in search results; sparse field cannot be output
    limit=num_retreive,
    search_params=search_params
)

for hits in iter(retreived_chunks):
    for hit in hits:
        print("-----------------------------------------")
        print(f"BM-25 score is {hit.entity.get('distance')}")
        print(f"Retreived text:\n{hit.entity.get('text')}")

-----------------------------------------
BM-25 score is 18.388214111328125
Retreived text:
To request an NYU HPC account please log in to
[NYU Identity Management service](https://identity.it.nyu.edu/)
and follow the link to "Request HPC account" by clicking on the hamburger icon on the top left of the page and selecting "Manage Access" . We have a walkthrough of how to
[request an account through IIQ](/docs/hpc/getting_started/HPC_Accounts/walkthrough_request_hpc_account)
. If you are a student, alumni or an external collaborator you will need an NYU faculty sponsor.
-----------------------------------------
BM-25 score is 18.080692291259766
Retreived text:
[Previous How to Approve an HPC Account Request](/docs/hpc/getting_started/HPC_Accounts/walkthrough_approve_hpc_account_request)
[Next HPC Accounts for Sponsored External Collaborators](/docs/hpc/getting_started/HPC_Accounts/hpc_accounts_external_collaborators)
-----------------------------------------
BM-25 score is 17.5222358703